# CV Kaggle Competition: object detection на 52 класса

В ноутбуке собран полный пайплайн решения соревнования: проверка данных, EDA, подготовка датасета для YOLO, обучение модели, инференс на test и постпроцессинг в формат Kaggle submission.

Итоговая модель: `YOLO11m`, дообученная на всём train.  
Формат предсказаний: `class confidence x1 y1 x2 y2`, координаты в пикселях.

Все seed зафиксированы как `993`.

## 1. Установка и проверка окружения

In [ ]:
import os
import random
import shutil
import zipfile
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

seed = 993

random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as e:
    print(e)

## 2. Загрузка данных

In [ ]:
work_dir = Path("/content/kaggle_cv")
zip_path = Path("/content/drive/MyDrive/2026-cv-competition.zip")

if not work_dir.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

    work_dir.mkdir(parents=True, exist_ok=True)

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(work_dir)
    else:
        raise FileNotFoundError("Не найден 2026-cv-competition.zip. Проверь путь к архиву или распакуй данные вручную.")

root = work_dir

train_images_dir = root / "train" / "train" / "images"
train_labels_dir = root / "train" / "train" / "labels"
test_images_dir = root / "test" / "test" / "images"
sample_path = root / "sample_submission.csv"

print("train images:", train_images_dir.exists())
print("train labels:", train_labels_dir.exists())
print("test images:", test_images_dir.exists())
print("sample:", sample_path.exists())

## 3. Первичная проверка датасета

In [ ]:
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

train_images = sorted([p for p in train_images_dir.iterdir() if p.suffix.lower() in img_exts])
train_labels = sorted([p for p in train_labels_dir.iterdir() if p.suffix.lower() == ".txt"])
test_images = sorted([p for p in test_images_dir.iterdir() if p.suffix.lower() in img_exts])

print("train images:", len(train_images))
print("train labels:", len(train_labels))
print("test images:", len(test_images))
print("first train image:", train_images[0].name)
print("first label:", train_labels[0].name)
print("first test image:", test_images[0].name)

In [ ]:
for p in train_labels[:3]:
    print("\n---", p.name, "---")
    print(p.read_text(encoding="utf-8")[:700])

Разметка уже находится в YOLO-формате: `class x_center y_center width height`, координаты нормализованы от 0 до 1.

## 4. EDA и проверка разметки

In [ ]:
rows = []
bad = []
empty = []

for label_path in train_labels:
    text = label_path.read_text(encoding="utf-8").strip()

    if not text:
        empty.append(label_path.name)
        continue

    for i, line in enumerate(text.splitlines(), 1):
        parts = line.split()

        if len(parts) != 5:
            bad.append((label_path.name, i, line))
            continue

        c, x, y, w, h = parts

        try:
            c = int(c)
            x, y, w, h = map(float, [x, y, w, h])
        except Exception:
            bad.append((label_path.name, i, line))
            continue

        rows.append({"file": label_path.name, "class": c, "x": x, "y": y, "w": w, "h": h})

labels_df = pd.DataFrame(rows)

print("objects:", len(labels_df))
print("images with labels:", labels_df["file"].nunique())
print("empty label files:", len(empty))
print("bad lines:", len(bad))
print("classes:", labels_df["class"].nunique())
print("min class:", labels_df["class"].min())
print("max class:", labels_df["class"].max())

print("\ncoordinate min/max:")
print(labels_df[["x", "y", "w", "h"]].agg(["min", "max"]))

print("\nobjects per class:")
print(labels_df["class"].value_counts().sort_index())

In [ ]:
sizes = []
items = []

for img_path in train_images:
    img = Image.open(img_path)
    w, h = img.size
    label_path = train_labels_dir / f"{img_path.stem}.txt"
    n = len(label_path.read_text(encoding="utf-8").strip().splitlines())

    sizes.append((w, h))
    items.append(n)

sizes_df = pd.DataFrame(sizes, columns=["width", "height"])
items_df = pd.DataFrame({"objects": items})

print("image sizes:")
print(sizes_df.value_counts().head())

print("\nobjects per image:")
print(items_df["objects"].describe())

plt.figure(figsize=(7, 4))
items_df["objects"].hist(bins=30)
plt.title("Objects per image")
plt.xlabel("objects")
plt.ylabel("images")
plt.show()

In [ ]:
def show_image_with_boxes(img_path):
    label_path = train_labels_dir / f"{img_path.stem}.txt"

    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    draw = ImageDraw.Draw(img)

    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        c, x, y, bw, bh = line.split()
        x, y, bw, bh = map(float, [x, y, bw, bh])

        x1 = (x - bw / 2) * w
        y1 = (y - bh / 2) * h
        x2 = (x + bw / 2) * w
        y2 = (y + bh / 2) * h

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(y1 - 15, 0)), c, fill="red")

    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

for img_path in random.sample(train_images, 3):
    print(img_path.name)
    show_image_with_boxes(img_path)

## 5. Train/val split для выбора модели

Для промежуточной оценки делался split 80/20. Стратификация построена по редкому классу на изображении, чтобы в val остались все классы.

In [ ]:
image_classes = {}

for img_path in train_images:
    label_path = train_labels_dir / f"{img_path.stem}.txt"
    classes = []

    for line in label_path.read_text(encoding="utf-8").strip().splitlines():
        classes.append(int(line.split()[0]))

    image_classes[img_path.name] = sorted(set(classes))

class_image_count = {}

for classes in image_classes.values():
    for c in classes:
        class_image_count[c] = class_image_count.get(c, 0) + 1

keys = []

for img_path in train_images:
    classes = image_classes[img_path.name]
    rare_class = min(classes, key=lambda c: class_image_count[c])
    keys.append(rare_class)

key_counts = pd.Series(keys).value_counts()
stratify = keys if key_counts.min() >= 2 else None

train_part, val_part = train_test_split(
    train_images,
    test_size=0.2,
    random_state=seed,
    shuffle=True,
    stratify=stratify
)

print("train:", len(train_part))
print("val:", len(val_part))


def count_classes(images):
    counts = {i: 0 for i in range(52)}

    for img_path in images:
        label_path = train_labels_dir / f"{img_path.stem}.txt"

        for line in label_path.read_text(encoding="utf-8").strip().splitlines():
            c = int(line.split()[0])
            counts[c] += 1

    return pd.Series(counts)

split_check = pd.DataFrame({
    "all": count_classes(train_images),
    "train": count_classes(train_part),
    "val": count_classes(val_part)
})

split_check["val_share"] = (split_check["val"] / split_check["all"]).round(3)
split_check

## 6. Подготовка YOLO-датасета для эксперимента на split

In [ ]:
yolo_dir = Path("/content/dataset_yolo")

if yolo_dir.exists():
    shutil.rmtree(yolo_dir)

for part in ["train", "val"]:
    (yolo_dir / "images" / part).mkdir(parents=True, exist_ok=True)
    (yolo_dir / "labels" / part).mkdir(parents=True, exist_ok=True)


def copy_split(images, part):
    for img_path in tqdm(images):
        label_path = train_labels_dir / f"{img_path.stem}.txt"
        shutil.copy2(img_path, yolo_dir / "images" / part / img_path.name)
        shutil.copy2(label_path, yolo_dir / "labels" / part / label_path.name)

copy_split(train_part, "train")
copy_split(val_part, "val")

data = {
    "path": str(yolo_dir),
    "train": "images/train",
    "val": "images/val",
    "nc": 52,
    "names": [str(i) for i in range(52)]
}

with open(yolo_dir / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("train images:", len(list((yolo_dir / "images" / "train").glob("*"))))
print("val images:", len(list((yolo_dir / "images" / "val").glob("*"))))
print("yaml:", yolo_dir / "data.yaml")

## 7. Обучение базовой модели на split

Модель `yolo11m.pt` использовалась как основной кандидат. На split она дала лучший результат среди быстрых проверок и затем была дообучена на всём train.

In [ ]:
model = YOLO("yolo11m.pt")

train_result = model.train(
    data=str(yolo_dir / "data.yaml"),
    epochs=70,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    seed=seed,
    deterministic=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    patience=18,
    cos_lr=True,
    close_mosaic=12,
    cache=True,
    project="/content/runs",
    name="yolo11m_640",
    exist_ok=True
)

## 8. Финальное обучение на всём train

После выбора модели финальный вариант обучался на всех 1697 размеченных train-картинках. Test-картинки при обучении не использовались.

Валидационная папка здесь нужна только для технического логирования Ultralytics, поэтому её метрики не считаются честной оценкой качества на Kaggle.

In [ ]:
all_dir = Path("/content/dataset_yolo_all")

if all_dir.exists():
    shutil.rmtree(all_dir)

(all_dir / "images" / "train").mkdir(parents=True, exist_ok=True)
(all_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)
(all_dir / "images" / "val").mkdir(parents=True, exist_ok=True)
(all_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)

for img_path in tqdm(train_images):
    label_path = train_labels_dir / f"{img_path.stem}.txt"
    shutil.copy2(img_path, all_dir / "images" / "train" / img_path.name)
    shutil.copy2(label_path, all_dir / "labels" / "train" / label_path.name)

for img_path in tqdm(val_part):
    label_path = train_labels_dir / f"{img_path.stem}.txt"
    shutil.copy2(img_path, all_dir / "images" / "val" / img_path.name)
    shutil.copy2(label_path, all_dir / "labels" / "val" / label_path.name)

data_all = {
    "path": str(all_dir),
    "train": "images/train",
    "val": "images/val",
    "nc": 52,
    "names": [str(i) for i in range(52)]
}

with open(all_dir / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_all, f, sort_keys=False)

print("all train images:", len(list((all_dir / "images" / "train").glob("*"))))
print("val for logging:", len(list((all_dir / "images" / "val").glob("*"))))

In [ ]:
base_weights = "/content/runs/yolo11m_640/weights/best.pt"

if not Path(base_weights).exists():
    base_weights = "yolo11m.pt"

model = YOLO(base_weights)

train_result = model.train(
    data=str(all_dir / "data.yaml"),
    epochs=35,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    seed=seed,
    deterministic=True,
    optimizer="AdamW",
    lr0=0.00012,
    lrf=0.01,
    weight_decay=0.0005,
    patience=0,
    cos_lr=True,
    close_mosaic=5,
    cache=True,
    project="/content/runs",
    name="yolo11m_all_final",
    exist_ok=True
)

## 9. Инференс и постпроцессинг

Kaggle-валидатор принимает формат:

`class confidence x1 y1 x2 y2`

где координаты `x1, y1, x2, y2` заданы в пикселях.  
Финальный файл сохраняется как `sample_submission.csv`.

In [ ]:
best_path = "/content/runs/yolo11m_all_final/weights/best.pt"
model = YOLO(best_path)

try:
    shutil.copy2(best_path, "/content/drive/MyDrive/yolo11m_all_final_best.pt")
except Exception:
    pass

results_full = model.predict(
    source=str(test_images_dir),
    imgsz=640,
    conf=0.001,
    iou=0.55,
    max_det=150,
    augment=True,
    device=0,
    verbose=False
)

sample = pd.read_csv(sample_path)


def make_submission(thr):
    pred_map = {}

    for r in results_full:
        image_id = Path(r.path).stem
        parts = []

        if r.boxes is not None and len(r.boxes) > 0:
            cls = r.boxes.cls.cpu().numpy().astype(int)
            conf = r.boxes.conf.cpu().numpy()
            boxes = r.boxes.xyxy.cpu().numpy()

            ids = np.where(conf >= thr)[0]

            if len(ids) == 0:
                ids = np.array([int(np.argmax(conf))])

            ids = ids[np.argsort(-conf[ids])]
            h, w = r.orig_shape

            for i in ids:
                x1, y1, x2, y2 = boxes[i]

                x1, x2 = sorted([float(x1), float(x2)])
                y1, y2 = sorted([float(y1), float(y2)])

                x1 = min(max(x1, 0), w)
                x2 = min(max(x2, 0), w)
                y1 = min(max(y1, 0), h)
                y2 = min(max(y2, 0), h)

                parts += [
                    str(cls[i]),
                    f"{float(conf[i]):.6f}",
                    f"{x1:.2f}",
                    f"{y1:.2f}",
                    f"{x2:.2f}",
                    f"{y2:.2f}"
                ]

        pred_map[image_id] = " ".join(parts)

    sub = sample.copy()
    sub["PredictionString"] = sub["image_id"].map(pred_map).fillna("")
    return sub

In [ ]:
thresholds = [0.001, 0.003, 0.005, 0.01, 0.03]
rows = []
submissions = {}

for thr in thresholds:
    sub = make_submission(thr)
    tag = str(thr).replace(".", "")
    path = f"/content/submission_all_yolo11m_img640_conf{tag}.csv"
    drive_path = f"/content/drive/MyDrive/submission_all_yolo11m_img640_conf{tag}.csv"

    sub.to_csv(path, index=False)

    try:
        sub.to_csv(drive_path, index=False)
    except Exception:
        pass

    cnt = sub["PredictionString"].apply(
        lambda x: 0 if str(x) == "" or str(x) == "nan" else len(str(x).split()) // 6
    )

    rows.append({
        "file": path,
        "thr": thr,
        "empty": int((cnt == 0).sum()),
        "total_boxes": int(cnt.sum()),
        "boxes_per_image": round(float(cnt.mean()), 2),
        "min_boxes": int(cnt.min()),
        "max_boxes": int(cnt.max())
    })

    submissions[thr] = sub

check = pd.DataFrame(rows)
check

In [ ]:
final_thr = 0.003
final_sub = submissions[final_thr]

final_path = "/content/sample_submission.csv"
final_sub.to_csv(final_path, index=False)

print("saved:", final_path)
print(final_sub.shape)
print(final_sub.head())

cnt = final_sub["PredictionString"].apply(
    lambda x: 0 if str(x) == "" or str(x) == "nan" else len(str(x).split()) // 6
)

print("empty:", int((cnt == 0).sum()))
print("total boxes:", int(cnt.sum()))
print("boxes per image:", round(float(cnt.mean()), 2))

## 10. Итог

В финальном решении использовались:

- YOLO11m;
- обучение на split для выбора модели;
- финальное дообучение на всём train;
- postprocessing в формате `class confidence x1 y1 x2 y2`;
- координаты в пикселях;
- итоговый confidence threshold: `0.003`.

Финальный файл создаётся в конце ноутбука:

`/content/sample_submission.csv`